# Real time Performance evaluation of the model

## Import libraries

In [1]:
import time
import tensorflow as tf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

In [3]:
import os, sys
processing_source_path = os.path.abspath('Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
from DataLoaderPipeline import FeaturesDataGenerator, scrapingHistoricalData
#from MarketDataCollector import scrapingHistoricalData


## Configuring the data parameters and adapters

In [4]:
SHD=scrapingHistoricalData()

# Lista de criptomoedas
#cryptos = ['BTC']

cryptos = ['BTC']
symbol = cryptos
last_timestamp = pd.Timestamp('2020-02-05 16:00:00')

# Escolha o intervalo
interval = '4h'

# Obtenha os dados históricos
cryptos_df = SHD.get_crypto_historical_data(cryptos, interval, '2024-01-01')

In [5]:
features_indicators=['SCP', 'RSI_14', 'Williams_R', 'MFI','MACD',
                     'EMA9', 'EMA20', 'EMA50', 'EMA100', 'EMA200', 'MA111', 'MA350',
                     'Bollinger_Bands_Upper', 'Bollinger_Bands_Middle', 'Bollinger_Bands_Lower',
                     'CCI',  'ROC', 'Chaikin_Money_Flow']

#pred_days = 17 # last
pred_days = 25
buy_sell_threshold=[0.05,-0.05]

lookback = 40# 

batch_size = 128
shuffle = True
data_augmentation =True

min_norm=-1
max_norm=1

trade=['Hold','Buy','Sell']

# model parameters
input_shape = (lookback, len(features_indicators))
n_classes = len(trade)

datatype='2D'

list_of_models =['CNN_MultiHead_2D']


In [6]:
dataGen_inference = FeaturesDataGenerator(cryptos_df,  datatype=datatype, lookback = lookback, 
                                          pred_days = pred_days, shuffle= False, batch_size=1, 
                                          selected_features = features_indicators, data_augmentation=False, 
                                          min_max_norm_features=[min_norm, max_norm])

self.pred_days 25
input data shape (2618, 40, 18)
output data shape (2618, 3)


### Load the model

In [7]:
from keras import backend as K
weighted_categorical_crossentropy_loss= dataGen_inference.weighted_categorical_crossentropy(np.ones(3))

def matthews_correlation_coefficient(y_true, y_pred):
    tp = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    tn = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    fp = K.sum(K.round(K.clip((1 - y_true) * y_pred, 0, 1)))
    fn = K.sum(K.round(K.clip(y_true * (1 - y_pred), 0, 1)))

    num = tp * tn - fp * fn
    den = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / K.sqrt(den + K.epsilon())

In [8]:
import tensorflow_addons as tfa
trained_best_models={}
for model_name in list_of_models:
    print(model_name)
    #checkpoint_filepath =f'models/model_{model_name}_stock_{ticker}_lookback_{lookback}'
    checkpoint_filepath =f'models/model_{model_name}_crypto_lookback_{lookback}_best'
    trained_best_models[f'{model_name}']=tf.keras.models.load_model(
        checkpoint_filepath,
        custom_objects={'loss': weighted_categorical_crossentropy_loss, 'matthews_correlation_coefficient': matthews_correlation_coefficient})

d:\Projetos_python\Time_Series_Forecast\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
d:\Projetos_python\Time_Series_Forecast\.venv\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.15.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not 

CNN_MultiHead_2D





## Realtime Testing

In [9]:
# Function to avoid redudant signals 
def generate_signals(signals):
    trade_signals = ['Hold']
    for i in range(1,len(signals)):
        if signals[i] == 'Buy' and signals[i-1] == 'Buy':
            trade_signals.append('Hold')
        elif signals[i] == 'Buy' and signals[i-1] == 'Hold':
            trade_signals.append('Buy')
        elif signals[i] == 'Sell' and signals[i-1] == 'Sell':
            trade_signals.append('Hold')
        elif signals[i] == 'Sell' and signals[i-1] == 'Hold':
            trade_signals.append('Sell')
        else:
            trade_signals.append('Hold')


    return trade_signals

In [10]:
# Set the interval and data type for the simulation
interval = '4h'
datatype = '2D'  # Adjust the data type
trade = ["Hold", "Buy", "Sell"]  # Define the trade options

# Initialize the simulation parameters
# Initialize the positions list, initial balance, and fees
positions = []
initial_balance = 100
balance = initial_balance
fees = 0.002  # Set the transaction fee


In [11]:
import pandas as pd
import os

def save_recommendation(model_name=str, crypto=str, data=None, recommendation=None, timestamp=None):
    # Define the column names
    columns = ['Date', 'Time', 'recommendation', 'Price', 'Position', 'Quantity']

    # Get the current timestamp
    if timestamp is None:
        import datetime
        timestamp = datetime.datetime.now()

    # Create the result dictionary
    result = {
        'Date': timestamp.date(),
        'Time': timestamp.time(),
        'recommendation': recommendation,
        'Price': data.loc[len(data)-1, 'Close'],  # Assuming data is a pandas DataFrame
    }

    # Create the file path
    file_path = f'Recommendations/{model_name}_{crypto}_recommendation.csv'

    # Check if the file exists
    if not os.path.exists(file_path):
        # Create a new DataFrame and save it to the file
        df = pd.DataFrame([result], columns=columns)
        df.to_csv(file_path, index=False)
    else:
        # Create a new DataFrame and append it to the existing file
        df = pd.DataFrame([result], columns=columns)
        df.to_csv(file_path, mode='a', header=False, index=False)

In [12]:



# Define a function to check and execute trades
def check_and_execute(symbol, last_timestamp, interval, balance=100):
    """
    Check and execute trades based on the model predictions.
    
    Parameters:
    symbol (str): The cryptocurrency symbol.
    last_timestamp (pd.Timestamp): The last timestamp.
    interval (str): The interval for the simulation.
    balance (float): The current balance.
    
    Returns:
    label_pred (np.array): The predicted labels.
    last_timestamp (pd.Timestamp): The updated last timestamp.
    balance (float): The updated balance.
    """
    
    # Get the current date and time
    today = datetime.today()
    
    # Collect historical data for the last 60 days
    window_days = today - timedelta(days=60)
    start_time = window_days.strftime('%Y-%m-%d')
    data = SHD.get_crypto_historical_data(symbol, interval, start_time)
    current_timestamp = data.loc[len(data)-1,'Date']
    
    # Update the symbol
    symbol = cryptos
    
    # Generate features for inference
    x_data_inference = dataGen_inference.comput_features(data, pred_days=0)
    x_data = dataGen_inference.apply_NomrMinmax(x_data_inference, min_norm, max_norm, axis=0)
    
    # Reshape the data for 2D input
    if datatype == '2D':
        x_data = np.transpose(x_data, [0, 2, 1]).reshape(-1, 1, input_shape[1], input_shape[0])
    
    # Load the trained model and make predictions
    model_name = f'CNN_MultiHead_{datatype}'
    label_pred = trained_best_models[model_name].predict(x_data)
    
    # Define the thresholds for the trade signals
    TH = [0.5, 0.75, 0.65]
    
    # Generate trade signals based on the predictions
    trade_signals = np.array([
        trade[np.argmax(prediction)] if np.max(prediction) > TH[np.argmax(prediction)] else trade[0]
        for prediction in label_pred
    ])
    
    # Generate signals
    trade_signals = generate_signals(trade_signals)
    
    # Print the suggested trades and timestamp
    print(f'Suggested trades: {trade_signals[-2:]} >> Timestamp: {current_timestamp}')
    
    # Check for new trade opportunities
    if last_timestamp != current_timestamp:
        # Check if it's a buy or sell signal
        if trade_signals[-2] != "Hold":
            # Check if it's a buy signal
            if trade_signals[-2] == "Buy":
                # Buy the cryptocurrency
                if balance > 0:  # Only buy if there's a balance
                    price = data.loc[len(data)-1, 'Close']
                    position_value = (balance * (1 - fees)) / price  # Calculate the position value
                    balance = 0  # Zero out the balance
                    positions.append((current_timestamp, price, position_value))
                    print(f'Bought at {current_timestamp} for {price} with {position_value} coins')
            # Check if it's a sell signal
            elif trade_signals[-2] == "Sell":
                # Sell the cryptocurrency
                if positions:  # Only sell if there are positions
                    position = positions.pop(0)
                    price = data.loc[len(data)-1, 'Close']
                    balance = position[2] * price * (1 - fees)  # Calculate the new balance
                    profit = balance - initial_balance
                    position_value = 0  # Zero out the position value
                    print(f'Sold at {current_timestamp} for {price} with balance {balance:.2f} and profit {profit:.2f}')
                else:
                    print(f'No positions to sell at {current_timestamp}')
        
        # Update the last timestamp and balance
        last_timestamp = current_timestamp

        # save the recommendations
        save_recommendation(model_name="CNN", crypto="BTC", data=data, recommendation=trade_signals[-2], timestamp=last_timestamp)

    return label_pred, last_timestamp, balance


In [ ]:
# Run the simulation in an infinite loop
while True:
    try:
        # Check and execute trades
        label_pred, last_timestamp, balance = check_and_execute(symbol, last_timestamp, interval, balance)
    except Exception as e:
        # Print any exceptions
        print(e)
    # Wait for 20 minutes (1200 seconds) before checking again
    time.sleep(1200)

11/11 [==============================] - 1s 11ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
Bought at 2025-04-20 12:00:00 for 84260.87 with 0.0011844169185530602 coins
11/11 [==============================] - 1s 25ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 0s 10ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 0s 19ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 0s 10ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 1s 46ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 0s 6ms/step
Suggested trades: ['Buy', 'Hold'] >> Timestamp: 2025-04-20 12:00:00
11/11 [==============================] - 0s 6ms/step
Suggested trades: 